# Customer Churn Prediction - Full Experiment

This notebook implements the complete SageMaker Production Guide in one place.
After validating here, we'll modularize into `part2-mlops/`.

## Steps:
1. Setup & Config
2. Data Generation (synthetic)
3. EDA
4. Preprocessing
5. Upload to S3
6. Train on SageMaker
7. Evaluate
8. Deploy Endpoint
9. Predict
10. Cleanup

## 1. Setup & Configuration

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
from pathlib import Path
import json

# === CONFIGURATION ===
# CHANGE THESE VALUES:
REGION = 'eu-west-1'
ROLE_ARN = 'arn:aws:iam::YOUR_ACCOUNT_ID:role/YOUR_SAGEMAKER_ROLE'  # <-- CHANGE THIS

# Session setup
boto_session = boto3.Session(region_name=REGION)
session = sagemaker.Session(boto_session=boto_session)
bucket = session.default_bucket()
prefix = 'churn-example'

print(f'SageMaker version: {sagemaker.__version__}')
print(f'Bucket: {bucket}')
print(f'Region: {REGION}')

## 2. Data Generation

We create synthetic telecom customer data. In your real project, replace this with a DB query or S3 read.

In [ ]:
np.random.seed(42)
n = 5000

data = pd.DataFrame({
    'customer_id': range(1, n + 1),
    'tenure_months': np.random.randint(1, 72, n),
    'monthly_charges': np.random.uniform(20, 100, n).round(2),
    'total_charges': np.random.uniform(100, 7000, n).round(2),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.35, 0.45, 0.2]),
    'online_security': np.random.choice(['Yes', 'No'], n),
    'tech_support': np.random.choice(['Yes', 'No'], n),
    'num_support_tickets': np.random.poisson(2, n),
})

# Target: churn (with realistic logic)
churn_prob = (
    0.3 * (data['contract_type'] == 'Month-to-month').astype(float) +
    0.2 * (data['tenure_months'] < 12).astype(float) +
    0.15 * (data['monthly_charges'] > 70).astype(float) +
    0.1 * (data['num_support_tickets'] > 3).astype(float) +
    np.random.uniform(0, 0.3, n)
)
data['churn'] = (churn_prob > 0.5).astype(int)

print(f'Shape: {data.shape}')
print(f'Churn rate: {data["churn"].mean():.2%}')
data.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.countplot(data=data, x='contract_type', hue='churn', ax=axes[0,0])
axes[0,0].set_title('Churn by Contract Type')

sns.histplot(data=data, x='tenure_months', hue='churn', kde=True, ax=axes[0,1])
axes[0,1].set_title('Tenure Distribution')

sns.boxplot(data=data, x='churn', y='monthly_charges', ax=axes[1,0])
axes[1,0].set_title('Monthly Charges by Churn')

sns.countplot(data=data, x='num_support_tickets', hue='churn', ax=axes[1,1])
axes[1,1].set_title('Support Tickets by Churn')

plt.tight_layout()
plt.show()

print('\nKey Observations:')
print(f'  - Month-to-month churn rate: {data[data["contract_type"]=="Month-to-month"]["churn"].mean():.2%}')
print(f'  - Two year churn rate: {data[data["contract_type"]=="Two year"]["churn"].mean():.2%}')
print(f'  - Short tenure (<12mo) churn: {data[data["tenure_months"]<12]["churn"].mean():.2%}')

## 4. Preprocessing

Key steps:
- Drop ID column
- Encode categoricals
- Put target column FIRST (SageMaker XGBoost requirement)
- Split into train/validation/test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Drop ID
df = data.drop('customer_id', axis=1)

# Encode categoricals
cat_cols = ['contract_type', 'payment_method', 'internet_service', 'online_security', 'tech_support']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Reorder: target first
target = 'churn'
features = [c for c in df.columns if c != target]
df_final = df[[target] + features]

# Split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(df_final, test_size=0.3, random_state=42, stratify=df_final[target])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df[target])

print(f'\nTrain: {train_df.shape}')
print(f'Val:   {val_df.shape}')
print(f'Test:  {test_df.shape}')

## 5. Upload to S3

In [ ]:
# Save locally first
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

train_df.to_csv(data_dir / 'train.csv', index=False, header=False)
val_df.to_csv(data_dir / 'validation.csv', index=False, header=False)
test_df.to_csv(data_dir / 'test.csv', index=False, header=False)

# Upload to S3
train_s3 = session.upload_data(str(data_dir / 'train.csv'), bucket=bucket, key_prefix=f'{prefix}/data/train')
val_s3 = session.upload_data(str(data_dir / 'validation.csv'), bucket=bucket, key_prefix=f'{prefix}/data/validation')
test_s3 = session.upload_data(str(data_dir / 'test.csv'), bucket=bucket, key_prefix=f'{prefix}/data/test')

print(f'Train: {train_s3}')
print(f'Val:   {val_s3}')
print(f'Test:  {test_s3}')

## 6. Train on SageMaker

This launches a training job on AWS infrastructure. Your laptop just submits the request.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# Get XGBoost container
xgb_image = sagemaker.image_uris.retrieve('xgboost', REGION, '1.7-1')
print(f'Image: {xgb_image}')

# Configure estimator
estimator = Estimator(
    image_uri=xgb_image,
    role=ROLE_ARN,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/{prefix}/models',
    sagemaker_session=session,
    base_job_name='churn-xgb'
)

# Set hyperparameters
estimator.set_hyperparameters(
    objective='binary:logistic',
    num_round=100,
    max_depth=5,
    eta=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc'
)

# Launch training (takes ~5 min: instance startup + training)
estimator.fit({
    'train': TrainingInput(s3_data=train_s3, content_type='text/csv'),
    'validation': TrainingInput(s3_data=val_s3, content_type='text/csv')
}, wait=True, logs='All')

print(f'\nModel artifact: {estimator.model_data}')

## 7. Evaluate

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Use Batch Transform to get predictions on test set
transformer = estimator.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/predictions'
)

# Prepare test data without target
test_features = test_df.iloc[:, 1:]
test_features.to_csv(data_dir / 'test_features.csv', index=False, header=False)
test_features_s3 = session.upload_data(
    str(data_dir / 'test_features.csv'), bucket=bucket, key_prefix=f'{prefix}/data/test_features'
)

transformer.transform(data=test_features_s3, content_type='text/csv', split_type='Line')
transformer.wait()
print('Batch transform complete!')

In [ ]:
# Download predictions and evaluate
import io

s3_client = boto3.client('s3', region_name=REGION)
pred_key = f'{prefix}/predictions/test_features.csv.out'
response = s3_client.get_object(Bucket=bucket, Key=pred_key)
preds_raw = response['Body'].read().decode('utf-8')
y_prob = np.array([float(x) for x in preds_raw.strip().split('\n')])

y_true = test_df.iloc[:, 0].values
y_pred = (y_prob > 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_true, y_prob)
print('=' * 50)
print('MODEL EVALUATION')
print('=' * 50)
print(f'\nAUC-ROC: {auc:.4f}')
print(f'\n{classification_report(y_true, y_pred, target_names=["No Churn", "Churn"])}')
print(f'Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}')

# Quality gate check
AUC_THRESHOLD = 0.75
if auc >= AUC_THRESHOLD:
    print(f'\n✅ PASSED quality gate (AUC {auc:.4f} >= {AUC_THRESHOLD})')
else:
    print(f'\n❌ FAILED quality gate (AUC {auc:.4f} < {AUC_THRESHOLD})')

## 8. Deploy Endpoint

In [ ]:
# Deploy real-time endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    serializer=sagemaker.serializers.CSVSerializer(),
    deserializer=sagemaker.deserializers.CSVDeserializer()
)

print(f'Endpoint: {predictor.endpoint_name}')

## 9. Test Predictions

In [ ]:
# Test with sample customers
samples = test_features.iloc[:5].values

print('Real-time predictions:')
print('-' * 60)
for i, row in enumerate(samples):
    result = predictor.predict(row.tolist())
    prob = float(result[0][0])
    actual = int(y_true[i])
    print(f'  Customer {i+1}: P(churn)={prob:.3f} → {"CHURN" if prob>0.5 else "STAY"} | Actual: {"CHURN" if actual==1 else "STAY"}')

## 10. Cleanup

⚠️ **IMPORTANT**: Delete the endpoint to stop charges!

In [ ]:
# UNCOMMENT TO DELETE (saves money!)
# predictor.delete_endpoint()
# print('Endpoint deleted.')

---

## ✅ Experiment Complete!

**What we proved:**
- Data can be processed and uploaded to S3
- XGBoost trains successfully on SageMaker
- Model passes quality gate (AUC > 0.75)
- Endpoint serves predictions in real-time

**Next step:** Open `../part2-mlops/` to see how this becomes production code.